# Example: Designing different divertor configurations

---

Here we show how to use the static inverse solver to make changes to the divertor legs, while holding the plasma core fixed. We make a few different divertor configurations, that correspond to larger flux expansion, connection lengths, and therefore heat fluxes. 
- conventional divertor (CD), as seen many time before in these notebooks. 
- elongated divertor (ED). 
- super-X divertor (SXD). 
- X-point target (XPT). 

We will check the equilibria vs our constraints and ensure that coil limits are not violated.

### Starting equilibrium

Build the MAST-U-like machine and solve for an initial free-boundary equilibrium using the static forward solver (with prescribed coil currents). 

In [ ]:
# Build the MAST-U-like machine.
import numpy as np
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

# Set up the equilibrium grid.
from freegsnke import equilibrium_update
from freegsnke.jtor_update import ConstrainPaxisIp

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=129,
)

# Set the plasma profile.
from freegsnke.jtor_update import Lao85
profiles = Lao85(
    eq=eq,
    Ip=744117,
    fvac=-0.47,
    alpha=np.array([418995.30508349,  10291.92267398]),
    beta=np.array([0.36624991, 0.05139343]),
    alpha_logic=True,
    beta_logic=True,    
)

# Set up the static Grad–Shafranov solver.
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq, gs_operator_order=4)


In [ ]:
# apply some starting PF coil currents
current_values = {
    'Solenoid': -9832.358940664852,
    'PX': 2146.99751691171,
    'D1': 4712.206686638823,
    'D2': 5311.404252957054,
    'D3': -1495.067768034757,
    'Dp': -3747.2015273389216,
    'D5': -1372.8402949519268,
    'D6': -337.80910404610916,
    'D7': 890.1958007147839,
    'P4': -3250.4415739978294,
    'P5': -5449.769306785791,
    'P6': -66.34934177326818,
}

for key in current_values.keys():
    eq.tokamak.set_coil_current(coil_label=key, current_value=current_values[key])

GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-7,
    verbose=True,
)

So here we have the standard CD equilibrium. 

In [ ]:
import matplotlib.pyplot as plt

# Plot the starting equilibrium.
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Elongated Divertor (ED)

Using the inverse solver, we can choose a set of isoflux constraints that will hold the core fixed, while moving the leg to the desired position (via the strikepoint).

First, we define a short function that takes in the contour defining the divertor and a radial co-ordinate, returning the corresponding Z co-ordinate. 

In [ ]:
from shapely.geometry import LineString

def Z_at_R(line, R):
    """Return all Z-values where a Shapely line crosses R."""
    
    _, Zmin, _, Zmax = line.bounds
    vertical = LineString([(R, Zmin - 1), (R, Zmax + 1)])
    intersection = line.intersection(vertical)

    if intersection.is_empty:
        return np.array([])

    if intersection.geom_type == "Point":
        return np.array([intersection.y])

    if intersection.geom_type == "MultiPoint":
        return np.array([point.y for point in intersection.geoms])

    raise ValueError(f"Line overlaps R={R}; Z is not unique.")

In [ ]:
# define the lower divertor chamber wall
lower_divertor_line = np.array(
    [[0.5345, -1.514], [1.0882, -2.0661], [1.3485, -2.0652], [1.7302, -1.6810]]
)
divertor_line = LineString(lower_divertor_line)

# choose location of the strikepoint
r_strike = 1.12
z_strike = Z_at_R(line=divertor_line, R=r_strike)
print(z_strike)

In [ ]:
# show the proposed new strikepoint
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=120)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.scatter(r_strike, z_strike, color='deeppink', marker='.', s=90, zorder=5)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, -1)
plt.tight_layout()

Now we define the constraints for the inverse problem.

In [ ]:
import numpy as np 
from freegsnke.inverse import Inverse_optimizer

# extract the X-points inside the first wall
xpts = eq.xpt[0:2,0:2]

# extract a few points on the plasma core boundary
LCFS = eq.separatrix(ntheta=12)

# define the null point constraints as the X-points
null_points = [[xpts[0,0], xpts[1,0]], [xpts[0,1], xpts[1,1]]]

# constrain the core and requested wall point to a common flux surface (i.e. the plasma separatrix)
# --> this is the plasma core, the upper X-point (as plasma is slightly upper single null), and the new strikepoint
isoflux_set = np.array([
    [
        np.hstack((LCFS[:,0], xpts[0,0], r_strike)).tolist(), 
        np.hstack((LCFS[:,1], xpts[0,1], z_strike)).tolist(),
    ]
])

# here we will hold the solenoid and P6 currents fixed
eq.tokamak['Solenoid'].control = False
eq.tokamak['P6'].control = False

# define the PF coil bounds for the coils currents being optimsied (upper, then lower)
# ordering: PX, D1, D2, D3, Dp, D5, D6, D7, P4, P5
coil_current_limits = [
    [5e3, 9e3, 9e3, 7e3, 7e3, 5e3, 5e3, 5e3, 0.0, 0.0],
    [-5e3, -9e3, -9e3, -7e3, -7e3, -5e3, -5e3, -5e3, -10e3, -10e3]
]

# initialise the inverse constraints
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
)

# default parameter for violating PF current limits (higher = stronger penalty)
constrain.mu_coils = 1e5

# regularisation vector (one for each optimised PF coil)
l2_reg = [1e-12]*10

In [ ]:
# save coil currents for comparison after solve
coil_names = [
    coil_name
    for coil_name in eq.tokamak.coils_dict
    if eq.tokamak[coil_name].control
]

currents = eq.tokamak.getCurrents()
active_currents_CD = np.array(
    [currents[name] for name in coil_names],
    dtype=float,
)

Now we can solve the inverse problem with weak current regularisation, then compare the equilibrium and currents with the CD case. 

In [ ]:
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=constrain, 
    target_relative_tolerance=1e-6,
    target_relative_psit_update=1e-3,
    verbose=True,
    l2_reg=l2_reg,
)

In [ ]:
# plot new equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

In [ ]:
# normalisation bounds for the controlled coils.
max_limits = np.asarray(coil_current_limits[0], dtype=float)
min_limits = np.asarray(coil_current_limits[1], dtype=float)
limit_width = max_limits - min_limits

# prep coil currents for plot
norm_current_CD = (
    2 * (active_currents_CD - min_limits) / limit_width - 1
)

currents = eq.tokamak.getCurrents()
active_currents_ED = np.array(
    [currents[name] for name in coil_names],
    dtype=float,
)

norm_current_ED = (
    2 * (active_currents_ED - min_limits) / limit_width - 1
)

x = np.arange(len(coil_names))

Now we can see how the coil currents have changed following the leg movement, all still within the bounds. 

In [ ]:
# plot
plt.figure(figsize=(12, 5), dpi=70)
plt.plot(x, norm_current_CD, "rx-", label="CD currents")
plt.plot(x, norm_current_ED, "bx-", label="ED currents")

plt.axhline(-1, color="k", label="Minimum limit")
plt.axhline(1, color="k", label="Maximum limit")

plt.xticks(x, coil_names, rotation=45, ha="right")
plt.ylabel("Normalised current")
plt.title("Controlled-coil current proximity to limits")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### Super-X Divertor (SXD)

Using this equilibirum, we can now push the leg further into the SXD configuration - this requires an additional isoflux constraint along the length of the leg. 

In [ ]:
# new strikepoint location
r_strike = 1.5
z_strike = Z_at_R(line=divertor_line, R=r_strike)

# nose isoflux constraint needed to ensure leg does not touch upper/lower tiles
nose_r = 0.72
nose_z = -1.55

# additional constraint along divertor leg
r_leg = 1.15
z_leg = -1.8

isoflux_set = np.array([
    [
        np.hstack((LCFS[:,0], xpts[0,0], nose_r, r_leg, r_strike)).tolist(), 
        np.hstack((LCFS[:,1], xpts[0,1], nose_z, z_leg, z_strike)).tolist(),
    ]
])

# add the extra X-point constraint
null_points = [[xpts[0,0], xpts[1,0]], [xpts[0,1], xpts[1,1]]]


# initialise the new constraints
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
)


In [ ]:
# show the proposed new strikepoint and leg constrain
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=120)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.scatter(nose_r, nose_z, color='deeppink', marker='.', s=90, zorder=5)
ax1.scatter(r_strike, z_strike, color='deeppink', marker='.', s=90, zorder=5)
ax1.scatter(r_leg, z_leg, color='deeppink', marker='.', s=90, zorder=5)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, -1)
plt.tight_layout()

In [ ]:
# solve
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=constrain, 
    target_relative_tolerance=1e-6,
    target_relative_psit_update=1e-3,
    verbose=True,
    l2_reg=l2_reg,
)

New we can see the new SXD configuration with coil currents. 

In [ ]:
# new SXD configuration
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

Notice, however, that we have a stray ("parasitic") X-point in the divertor chamber. This pushes flux towards the upper part of the chamber, which in many cases can trip machine protection probes. 

To fix this, we can push the X-point away by enforcing some constraints on the normalised flux, i.e. smooth out the poloidal flux lines near the separatrix leg.

In [ ]:
# psi norm constraints to get rid of parasitic null point in divertor

# normalised psi constraints set with format:
# [R, Z, psiN, 1] --> ψ̂(R,Z) ≥ psiN  (the +1 or -1 defines ≥ or ≤)
# [R, Z, psiN, -1] --> ψ̂(R,Z) ≤ psiN
psi_norm_limits = [
    [1.5, -1.75, 1.15, 1],
    [1.5, -1.75, 1.2, -1],
]


# initialise the new constraints
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
    psi_norm_limits=psi_norm_limits,
    mu_psi_norm=1e7, # penalise how much the normalised psi constraint is violated (default 1e6)
)


In [ ]:
# show the location of the psi_norm constraint
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=120)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.scatter(nose_r, nose_z, color='deeppink', marker='.', s=90, zorder=5)
ax1.scatter(r_strike, z_strike, color='deeppink', marker='.', s=90, zorder=5)
ax1.scatter(r_leg, z_leg, color='deeppink', marker='.', s=90, zorder=5)
ax1.scatter(psi_norm_limits[0][0], psi_norm_limits[0][1], color='brown', marker='.', s=90, zorder=5)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, -1)
plt.tight_layout()

In [ ]:
# solve
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=constrain, 
    target_relative_tolerance=1e-6,
    target_relative_psit_update=1e-3,
    verbose=True,
    l2_reg=l2_reg,
)

In [ ]:
for psi_con in psi_norm_limits:
    sign = ">=" if psi_con[3] >= 0 else "<="
    print(f"Psi norm at ({psi_con[0]}, {psi_con[1]}) = {eq.psiNRZ(psi_con[0], psi_con[1]):.3f} ({sign} {psi_con[2]})")

In [ ]:
# new SXD configuration
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

In [ ]:
# prep coil currents for plot
currents = eq.tokamak.getCurrents()
active_currents_SXD = np.array(
    [currents[name] for name in coil_names],
    dtype=float,
)

norm_current_SXD = (
    2 * (active_currents_SXD - min_limits) / limit_width - 1
)

x = np.arange(len(coil_names))

Notice that DP is getting close to the limits! We will monitor this in the next configuation. 

In [ ]:
# plot 
plt.figure(figsize=(12, 5), dpi=70)
plt.plot(x, norm_current_ED, "rx-", label="ED currents")
plt.plot(x, norm_current_SXD, "bx-", label="SXD currents")

plt.axhline(-1, color="k", label="Minimum limit")
plt.axhline(1, color="k", label="Maximum limit")

plt.xticks(x, coil_names, rotation=45, ha="right")
plt.ylabel("Normalised current")
plt.title("Controlled-coil current proximity to limits")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

### X-point Targer Divertor (XPT)

Next, we can try to place an X-point along the divertor leg. This requires an additional null point constraint and one more isoflux constraint.

In [ ]:
# location for the X-point in the lower divertor
xpt_r = 1.38
xpt_z = -1.83

# add this new point in the null constraints
null_points = [[xpts[0,0], xpts[1,0], xpt_r], [xpts[0,1], xpts[1,1], xpt_z]]

# we now have the new x-point and nose constraints alongside the core constraints
isoflux_set = np.array([
    [
        np.hstack((LCFS[:,0], xpts[0,0], xpt_r, nose_r)).tolist(), 
        np.hstack((LCFS[:,1], xpts[0,1], xpt_z, nose_z)).tolist(), 
    ]
])

# initialise
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    coil_current_limits=coil_current_limits,
)


In [ ]:
# solve
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=constrain, 
    target_relative_tolerance=1e-6,
    target_relative_psit_update=1e-3,
    verbose=True,
    l2_reg=l2_reg,
)

Now we can see the new divertor leg passing nicely into the divertor chamber and ending at the new X-point. 

In [ ]:
# plot
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
constrain.plot(axis=ax1,show=True)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

In [ ]:
# prep coil currents for plot
currents = eq.tokamak.getCurrents()
active_currents_SXD = np.array(
    [currents[name] for name in coil_names],
    dtype=float,
)

norm_current_XPT = (
    2 * (active_currents_SXD - min_limits) / limit_width - 1
)

x = np.arange(len(coil_names))

Both DP and D7 are appoaching their limits, so we could have instead made use of the `l2_reg` parameter if need to hold these coil currents more fixed (i.e. we would increase the regularisation on those coils).

In [ ]:
# plot
plt.figure(figsize=(12, 5), dpi=70)
plt.plot(x, norm_current_SXD, "rx-", label="SXD currents")
plt.plot(x, norm_current_XPT, "bx-", label="XPT currents")

plt.axhline(-1, color="k", label="Minimum limit")
plt.axhline(1, color="k", label="Maximum limit")

plt.xticks(x, coil_names, rotation=45, ha="right")
plt.ylabel("Normalised current")
plt.title("Controlled coil current proximity to limits")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()